# 기계학습기초 팀 프로젝트

## 02. 전처리 및 파이프라인 구성

- 결측·비해당 코드 처리 → 타깃 생성 → 피처 엔지니어링
- Train/Test Split () 및 sklearn Pipeline 설정


---

# 0. 프로젝트 개요

## 0.1 연구 배경

본 프로젝트는 국민건강영양조사 제9기 1차년도(2022년) 자료를 활용하여 수면시간, 청각 문제, 소음 노출 관련 요인과 우울 위험군 사이의 관계를 분석한다.

## 0.2 문제 정의

- 입력 변수: 수면시간, 청각 상태, 직업적 소음 노출, 이명, 이어폰 소음 노출, 청각 활동제한 관련 변수
- 타깃 변수: PHQ-9 총점 기반 우울 위험군 여부
- 문제 유형: 지도학습 기반 이진 분류

## 0.3 분석 목표

- PHQ-9 총점 10점 이상 여부를 기준으로 우울 위험군 예측
- 수면 및 청각 관련 변수의 예측 기여도 확인
- 여러 분류 모델의 성능 비교
- 변수 중요도 및 도메인 관점 해석

## 0.4 분석 시 주의점

본 프로젝트에서는 이미 하나의 파일로 병합된 KNHANES 데이터를 사용하므로 별도의 데이터 병합 과정은 수행하지 않는다. 또한 PHQ-9 총점에는 수면 관련 문항이 포함되어 있으므로, 수면시간 변수와 우울 위험군 타깃 사이에 개념적 중복 가능성이 있음을 한계점에서 함께 해석한다.

---

# 1. 라이브러리 및 기본 설정

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

---

# 2. 데이터 불러오기

## 2.1 사용 데이터

국민건강영양조사 제9기 1차년도(2022년) 자료 중 이미 병합되어 있는 단일 데이터셋을 사용한다.

아래 코드에서 `file_path`만 실제 파일명에 맞게 수정하면 된다.

In [2]:
# 사용 중인 파일명에 맞게 수정하세요.
# 예: "hn22_all.sas7bdat", "final_dataset.csv", "knhanes_2022.csv"
file_path = "../data/raw/hn22_all.sas7bdat"

if file_path.endswith(".sas7bdat"):
    df_raw = pd.read_sas(file_path, format="sas7bdat")
elif file_path.endswith(".csv"):
    df_raw = pd.read_csv(file_path)
elif file_path.endswith((".xlsx", ".xls")):
    df_raw = pd.read_excel(file_path)
else:
    raise ValueError("지원하지 않는 파일 형식입니다. sas7bdat, csv, xlsx 중 하나를 사용하세요.")

# SAS 파일에서 문자형 변수가 bytes로 읽히는 경우 디코딩
for col in df_raw.columns:
    if df_raw[col].dtype == "object":
        df_raw[col] = df_raw[col].apply(
            lambda x: x.decode("cp949", errors="ignore") if isinstance(x, bytes) else x
        )

print("데이터 크기:", df_raw.shape)
df_raw.head()

데이터 크기: (6265, 685)


,mod_d,ID,ID_fam,year,region,town_t,apt_t,psu,sex,age,age_month,incm,ho_incm,incm5,ho_incm5,edu,occp,wt_hs,wt_itvex,wt_oe,wt_bia,wt_ntr,wt_tot,wt_oent,wt_biant,kstrata,cfam,genertn,allownc,house,live_t,ainc_unit1,ainc_1,ainc,marri_1,marri_2,fam_rela,tins,npins,ID_F,ID_M,D_1_1,D_2_1,D_2_wk,DI1_dg,DI1_ag,DI1_pr,DI1_pt,DI1_2,DI2_dg,...,Y_MLK_ST,Y_WN_ST,Y_SUP_YN,Y_SUP_KD1,Y_SUP_KD3,Y_SUP_KD4,Y_SUP_KD7,N_DIET,N_DIET_WHY,N_DT_DS,N_DT_ETC,N_WAT_C,N_BF,N_BFD_Y,N_DAY,N_INTK,N_EN,N_WATER,N_PROT,N_FAT,N_SFA,N_MUFA,N_PUFA,N_N3,N_N6,N_CHOL,N_CHO,N_TDF,N_SUGAR,N_CA,N_PHOS,N_NA,N_K,N_MG,N_FE,N_ZN,N_VA_RAE,N_VITD,N_VITE,N_CAROT,N_RETIN,N_B1,N_B2,N_NIAC,N_FOLATE,N_VITC,LF_secur_y,LF_BUYER,LF_SAFE,N_DUSUAL
0,2026.02.27.,YA01220302,YA012203,2022.0,1.0,1.0,2.0,YA01,2.0,56.0,NaN,3.0,4.0,4.0,4.0,3.0,7.0,7834.391511,9090.322425,10783.493262,11291.368622,10346.361689,10343.746528,11265.895141,11741.501904,101.0,4.0,4.0,20.0,2.0,2.0,1.0,9300.0,775.0,1.0,1.0,2.0,10.0,1.0,NaN,NaN,3.0,2.0,88.0,1.0,48.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,3.5,2.0,8.0,수요일,849.465975,1460.144833,513.082135,58.852416,28.463777,8.017655,9.421072,7.647123,0.882515,6.761384,40.991353,238.931003,29.240249,44.705424,498.323785,1003.351910,1273.050326,2480.515348,371.132271,10.380525,10.240582,454.396392,2.150000,7.365415,853.187240,383.340000,0.814587,1.182300,10.042185,291.264557,32.441192,1.0,NaN,1.0,3
1,2026.02.27.,YA01220303,YA012203,2022.0,1.0,1.0,2.0,YA01,1.0,30.0,NaN,3.0,4.0,3.0,4.0,4.0,3.0,7834.391511,13042.142562,20078.947634,19822.942221,19763.950889,19666.742049,21352.961071,20820.115372,101.0,4.0,4.0,20.0,2.0,2.0,1.0,9300.0,775.0,2.0,88.0,3.0,10.0,1.0,NaN,YA01220302,2.0,2.0,88.0,0.0,888.0,8.0,8.0,8.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,14.9,8.0,8.0,수요일,1441.052658,1971.535569,998.779824,98.998643,59.509129,14.244400,25.674881,11.391725,0.802900,10.573850,361.378293,268.663768,47.771905,102.638985,742.705707,1692.230686,1968.873043,3676.696548,520.241317,19.408543,14.736918,815.571495,5.498400,11.608507,973.767210,732.385000,1.829417,2.017004,18.342991,329.385371,79.903529,1.0,NaN,1.0,3
2,2026.02.27.,YA01220304,YA012203,2022.0,1.0,1.0,2.0,YA01,2.0,25.0,NaN,3.0,4.0,4.0,4.0,4.0,7.0,7834.391511,9895.727513,14566.872199,14931.827386,14314.436646,14510.887963,15616.336412,15501.618290,101.0,4.0,4.0,20.0,2.0,2.0,1.0,9300.0,775.0,2.0,88.0,3.0,10.0,1.0,NaN,YA01220302,2.0,1.0,3.0,0.0,888.0,8.0,8.0,8.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,8.0,NaN,NaN,10.0,2.0,8.0,수요일,377.138748,1164.875037,131.628999,22.740898,39.650722,13.289250,15.123441,8.740357,0.354360,7.035220,0.961147,176.112561,10.117903,26.400092,423.150590,300.822929,1498.745141,783.549944,105.706399,2.591360,2.285068,14.093456,0.000000,8.502485,166.686411,0.000000,0.262020,1.379288,3.042505,69.694336,2.776080,1.0,1.0,1.0,2
3,2026.02.27.,YA01236501,YA012365,2022.0,1.0,1.0,2.0,YA01,1.0,66.0,NaN,3.0,3.0,4.0,3.0,4.0,7.0,7834.391511,6438.920648,7251.845005,7542.480144,7068.650323,7051.216076,7843.924333,8088.513725,101.0,2.0,2.0,20.0,2.0,2.0,1.0,4800.0,400.0,1.0,1.0,1.0,20.0,1.0,NaN,NaN,2.0,2.0,88.0,0.0,888.0,8.0,8.0,8.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,8.0,NaN,NaN,5.0,8.0,8.0,화요일,3245.543916,2213.970700,2744.107388,91.387506,66.330671,16.131310,23.075432,18.333469,3.835584,14.137622,325.006948,321.141632,48.315870,65.847307,552.891309,1441.801023,4157.860611,4448.107487,480.592319,16.433790,17.301592,407.893533,3.230366,9.215554,3676.549476,100.898983,1.338621,2.325439,18.325182,633.687278,112.775603,1.0,NaN,1.0,2
4,2026.02.27.,YA01236502,YA012365,2022.0,1.0,1.0,2.0,YA01,2.0,62.0,NaN,3.0,3.0,3.0,3.0,3.0,6.0,7834.391511,5593.752658,6857.115069,7064.391215,6434.393479,6420.343834,7194.949559,7421.746687,101.0,2.0,2.0,20.0,2.0,2.0,1.0,4800.0,400.0,1.0,1.0,2.0,20.0,1.0,NaN,NaN,2.0,2.0,88.0,0.0,888.0,8.0,8.0,8.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,8.0,NaN,NaN,5.0,2.0,8.0,화요일,2864.424520,2062.847547,2392.725794,93.020540,64.847477,16.525857,24.244867,15.025077,3.119590,1

---

# 3. 데이터 구조 확인 및 분석 변수 정리

이 단계에서는 병합을 수행하지 않고, 이미 병합된 데이터셋에서 분석에 필요한 변수들이 존재하는지 확인한다.

## 3.1 주요 변수 영역

- 수면시간: `BP16_1`, `BP16_2`
- 청력 자가보고: `T_Q_HR`
- 청각보조기기: `T_Q_HR_1`, `T_Q_HR_2`
- 직업적 소음 노출: `T_NQ_OCP`
- 이어폰 소음 노출: `T_NQ_PH2`, `T_NQ_PH2_T`
- 이명 경험: `T_Q_VN`, `T_Q_VN_1`, `T_Q_VN_2`
- 활동제한: `LQ4_00`, `LQ4_13`
- PHQ-9 문항: `BP_PHQ_1` ~ `BP_PHQ_9`
- PHQ-9 총점: `mh_PHQ_S`

In [3]:
# 사용하는 열 코드북
USE_COLS = {
    # ID
    "ID": "id",

    # Sleep
    "BP16_1": "wkdy_sleep_hours",      # 주중 하루 평균 수면시간
    "BP16_2": "wknd_sleep_hours",      # 주말 하루 평균 수면시간

    # Hearing
    "T_Q_HR": "hear_status",           # 본인 청력 상태
    "T_Q_HR_1": "hear_device",         # 청각보조기기 사용 종류
    "T_Q_HR_2": "hear_device_freq",    # 청각보조기기 사용 빈도

    # Noise exposure
    "T_NQ_OCP": "occ_noise",           # 직업적 소음 노출
    "T_NQ_PH2": "ear_noise",           # 시끄러운 장소에서 이어폰 사용 경험
    "T_NQ_PH2_T": "ear_noise_min",     # 이어폰 사용 시간

    # Tinnitus
    "T_Q_VN": "tinnitus",              # 이명 경험
    "T_Q_VN_1": "tinnitus_6mo",        # 이명 6개월 이상 지속 여부
    "T_Q_VN_2": "tinnitus_dist",       # 이명 괴로움 수준

    # Activity limitation
    "LQ4_00": "act_limit",             # 활동제한 여부
    "LQ4_13": "hear_act_limit",        # 청각 문제로 인한 활동제한

    # PHQ-9 items
    "BP_PHQ_1": "phq_1",
    "BP_PHQ_2": "phq_2",
    "BP_PHQ_3": "phq_3",
    "BP_PHQ_4": "phq_4",
    "BP_PHQ_5": "phq_5",
    "BP_PHQ_6": "phq_6",
    "BP_PHQ_7": "phq_7",
    "BP_PHQ_8": "phq_8",
    "BP_PHQ_9": "phq_9",

    # PHQ-9 total score
    "mh_PHQ_S": "phq_score",
}

required_cols = list(USE_COLS.keys())
missing_cols = [col for col in required_cols if col not in df_raw.columns]
existing_cols = [col for col in required_cols if col in df_raw.columns]

print("분석 후보 변수 수:", len(required_cols))
print("존재하는 변수 수:", len(existing_cols))
print("누락 변수 수:", len(missing_cols))

if missing_cols:
    print("누락 변수:", missing_cols)
else:
    print("모든 분석 후보 변수가 데이터에 존재합니다.")

분석 후보 변수 수: 24
존재하는 변수 수: 24
누락 변수 수: 0
모든 분석 후보 변수가 데이터에 존재합니다.


In [4]:
# 존재하는 변수만 우선 선택한다.
# 단, 핵심 변수 누락이 있다면 코드북 또는 실제 컬럼명을 확인해야 한다.
df_selected = df_raw[existing_cols].rename(columns={k: v for k, v in USE_COLS.items() if k in existing_cols}).copy()

print(df_selected.shape)
df_selected.head()

(6265, 24)


,id,wkdy_sleep_hours,wknd_sleep_hours,hear_status,hear_device,hear_device_freq,occ_noise,ear_noise,ear_noise_min,tinnitus,tinnitus_6mo,tinnitus_dist,act_limit,hear_act_limit,phq_1,phq_2,phq_3,phq_4,phq_5,phq_6,phq_7,phq_8,phq_9,phq_score
0,YA01220302,8.0,10.0,1.0,8.0,8.0,2.0,2.0,8888.0,2.0,8.0,88.0,1.0,0.0,0.0,2.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0,7.0
1,YA01220303,7.0,7.0,1.0,8.0,8.0,8.0,2.0,8888.0,8.0,8.0,88.0,2.0,8.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,6.0
2,YA01220304,6.0,8.0,1.0,8.0,8.0,8.0,1.0,20.0,8.0,8.0,88.0,2.0,8.0,0.0,1.0,3.0,3.0,2.0,0.0,1.0,0.0,1.0,11.0
3,YA01236501,9.0,9.0,1.0,8.0,8.0,2.0,2.0,8888.0,2.0,8.0,88.0,2.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,YA01236502,7.0,9.0,1.0,8.0,8.0,2.0,2.0,8888.0,2.0,8.0,88.0,2.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---

# 4. 결측 및 비해당 코드 처리

KNHANES 설문 변수에는 비해당, 모름, 무응답 등의 코드가 포함될 수 있다. 분석 전에 이러한 값을 결측으로 처리한다.

주의: 변수별 코드북에 따라 결측 코드의 의미가 다를 수 있으므로, 최종 분석 전에는 반드시 코드북을 확인해야 한다. 아래 코드는 프로젝트 진행을 위한 기본 처리 예시이다.

## 전처리 결정 근거 (EDA 기반)

| 변수 | 처리 방법 | 근거 |
|------|-----------|------|
| `hear_device_freq` | missing indicator 생성 후 원본 제거 | 결측률 **98.5%** — 대부분 비해당자로 결측 자체가 정보를 가짐 |
| `wknd_sleep_hours` | `clip(2, 16)` 이상치 클리핑 | IQR 기준 이상치 비율 **21.3%** — 수면시간 물리적 범위 적용 |
| `wkdy_sleep_hours` | `clip(2, 16)` 이상치 클리핑 | 동일 기준 적용 (일관성 유지) |
| `act_limit` | 결측 median 대체 (파이프라인) | 타깃 상관 1위 (r = −0.209) — 제거하지 않고 유지 |

In [ ]:
# object 타입 컬럼만 숫자 변환 시도 (ID 등 문자형 식별자 제외)
# pandas 최신버전에서 errors="ignore"가 제거되어 select_dtypes로 대체
for col in df_selected.select_dtypes(include="object").columns:
    if col != "id":
        df_selected[col] = pd.to_numeric(df_selected[col], errors="coerce")

# 분석 변수별 결측 코드 처리 (코드북 기준)
missing_code_map = {
    "wkdy_sleep_hours":  [88, 99, 888, 999],
    "wknd_sleep_hours":  [88, 99, 888, 999],
    "hear_status":       [8, 9, 88, 99],
    "hear_device":       [8, 9, 88, 99],
    "hear_device_freq":  [8, 9, 88, 99],
    "occ_noise":         [8, 9, 88, 99],
    "ear_noise":         [8, 9, 88, 99],
    "ear_noise_min":     [888, 999, 8888, 9999],
    "tinnitus":          [8, 9, 88, 99],
    "tinnitus_6mo":      [8, 9, 88, 99],
    "tinnitus_dist":     [8, 9, 88, 99],
    "act_limit":         [8, 9, 88, 99],
    "hear_act_limit":    [8, 9, 88, 99],
    "phq_1": [8, 9], "phq_2": [8, 9], "phq_3": [8, 9],
    "phq_4": [8, 9], "phq_5": [8, 9], "phq_6": [8, 9],
    "phq_7": [8, 9], "phq_8": [8, 9], "phq_9": [8, 9],
    "phq_score": [88, 99, 888, 999],
}

for col, codes in missing_code_map.items():
    if col in df_selected.columns:
        df_selected[col] = df_selected[col].replace(codes, np.nan)

# 수면시간 물리적으로 불가능한 값 처리
for col in ["wkdy_sleep_hours", "wknd_sleep_hours"]:
    if col in df_selected.columns:
        df_selected.loc[(df_selected[col] < 0) | (df_selected[col] > 24), col] = np.nan

print("결측 처리 후 데이터 크기:", df_selected.shape)
df_selected.isna().mean().sort_values(ascending=False).head(20)

---

# 5. 타깃 변수 생성: PHQ-9 기반 우울 위험군

PHQ-9 총점이 10점 이상이면 우울 위험군으로 정의한다.

- 우울 위험군: `1`
- 비위험군: `0`

이 노트북에서는 `mh_PHQ_S`에서 가져온 `phq_score`를 우선 사용한다. `phq_score`가 없는 경우에는 PHQ-9 문항 9개를 합산하여 총점을 계산한다.

In [ ]:
# PHQ-9에서 3번 문항(수면 문항) 제외
# → 수면시간 변수(BP16_1, BP16_2)와 개념적 중복 방지 (데이터 누수)
phq_items_no3 = [f"phq_{i}" for i in range(1, 10) if i != 3]  # phq_3 제외

# 각 PHQ 문항 유효 범위 처리: 0~3만 유효, 그 외 → NaN
for col in phq_items_no3:
    if col in df_selected.columns:
        df_selected[col] = df_selected[col].where(df_selected[col].between(0, 3), np.nan)

# PHQ-8 총점 계산: 8문항 중 하나라도 결측이면 총점 NaN (skipna=False)
available_items = [col for col in phq_items_no3 if col in df_selected.columns]
df_selected["phq8_score"] = df_selected[available_items].sum(axis=1, skipna=False)

# 우울 위험군 정의: PHQ-8 총점 7점 이상 → 1, 미만 → 0, 결측 → NaN
df_selected["depression_risk"] = np.where(
    df_selected["phq8_score"].isna(), np.nan,
    (df_selected["phq8_score"] >= 7).astype(int)
)

print("=== PHQ-8 총점 기술 통계 (phq_3 수면 문항 제외) ===")
print(df_selected["phq8_score"].describe().round(2))
print(f"\n결측 (8문항 중 1개 이상 무응답): {df_selected['phq8_score'].isna().sum()}명")

print("\n=== 타깃 변수 분포 ===")
print(df_selected["depression_risk"].value_counts(dropna=False))
print(df_selected["depression_risk"].value_counts(normalize=True, dropna=False).round(3))

---

# 6. 모델링 데이터 구성

PHQ-9 문항과 PHQ-9 총점은 타깃 생성에 사용된 변수이므로 모델 입력 변수에서는 제외한다. 식별자인 `id` 역시 모델 입력에 포함하지 않는다.

In [ ]:
FEATURE_COLS = [
    "wkdy_sleep_hours",
    "wknd_sleep_hours",
    "hear_status",
    "hear_device",
    "hear_device_freq",   # 결측률 98.5% → missing indicator 생성 후 원본 제거
    "occ_noise",
    "ear_noise",
    "ear_noise_min",
    "tinnitus",
    "tinnitus_6mo",
    "tinnitus_dist",
    "act_limit",          # 타깃 상관 1위 (r=-0.209) → 유지
    "hear_act_limit",
]

available_features = [col for col in FEATURE_COLS if col in df_selected.columns]
model_cols = [col for col in ["id"] + available_features + ["depression_risk"] if col in df_selected.columns]

df_model = df_selected[model_cols].copy()
df_model = df_model.dropna(subset=["depression_risk"])
df_model["depression_risk"] = df_model["depression_risk"].astype(int)

# ── EDA 기반 전처리 1: hear_device_freq missing indicator 생성 (결측률 98.5%) ──
df_model["hear_device_freq_missing"] = df_model["hear_device_freq"].isna().astype(int)

# ── EDA 기반 전처리 2: 수면시간 이상치 클리핑 (wknd 21.3%, 물리적 범위 2~16시간) ──
for col in ["wkdy_sleep_hours", "wknd_sleep_hours"]:
    df_model[col] = df_model[col].clip(lower=2, upper=16)

print("모델링 데이터 크기:", df_model.shape)
df_model.head()

## 6.1 피처 엔지니어링

EDA 결과(수면시간 위험군 차이 유의)를 반영하여 수면 관련 파생 피처를 추가한다.

- `sleep_short` : 주중 수면시간 6시간 미만 이진 지시자 — 수면 부족 여부
- `sleep_diff` : 주말 수면시간 − 주중 수면시간 — 주중/주말 수면 패턴 변화

In [ ]:
# 피처 엔지니어링: 수면 관련 파생 피처 (클리핑 후 적용)
df_model["sleep_short"] = (df_model["wkdy_sleep_hours"] < 6).astype(int)
df_model["sleep_diff"]  = df_model["wknd_sleep_hours"] - df_model["wkdy_sleep_hours"]

print("추가된 피처: sleep_short, sleep_diff")
print(df_model[["wkdy_sleep_hours", "wknd_sleep_hours", "sleep_short", "sleep_diff"]].describe().round(2))

---

# 7. EDA

모델링 전에 타깃 변수 분포, 결측률, 주요 변수와 우울 위험군의 관계를 확인한다.

## EDA 요약

| # | 항목 | 결과 | 전처리 대응 전략 |
|---|------|------|-----------------|
| 1 | 클래스 불균형 | 13.7 : 1 (비위험군 : 위험군) | `stratify=y`, `class_weight='balanced'` |
| 2 | 수면시간 | 위험군이 더 짧은 경향 (p < 0.05, Mann-Whitney) | 수면 관련 파생 피처 추가 |
| 3 | 결측치 최다 | `hear_device_freq` (98.5%) | missing indicator 생성 후 원본 제거 |
| 4 | 타깃 상관 1위 | `act_limit` (r = −0.209) | 주요 피처로 유지 |
| 5 | 이상치 최다 | `wknd_sleep_hours` (21.3%) | `clip(2, 16)` 클리핑 |

In [ ]:
import seaborn as sns
from scipy import stats

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
os.makedirs("figures", exist_ok=True)

TARGET    = "depression_risk"
CONT_COLS = ["wkdy_sleep_hours", "wknd_sleep_hours", "ear_noise_min", "tinnitus_dist"]
CAT_COLS  = [c for c in available_features if c not in CONT_COLS]
colors    = ["steelblue", "tomato"]

print("=" * 60)
print(f"  EDA  |  샘플: {len(df_model)}  |  변수: {len(available_features)}")
print("=" * 60)

# ── 7.1 타깃 변수 분포 ────────────────────────────────────────────
counts    = df_model[TARGET].value_counts().sort_index()
imbalance = counts[0] / counts[1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["비위험군(0)", "위험군(1)"], counts.values, color=colors, edgecolor="white", width=0.5)
axes[0].set_title("우울 위험군 빈도")
axes[0].set_ylabel("샘플 수")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 15, str(v), ha="center", fontsize=11, fontweight="bold")
axes[1].pie(counts.values, labels=["비위험군(0)", "위험군(1)"],
            autopct="%1.1f%%", colors=colors, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("우울 위험군 비율")
plt.suptitle("7.1 타깃 변수 분포", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/01_target_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"클래스 불균형 비율: {imbalance:.1f}:1  →  {'⚠ 불균형 주의 (>5:1)' if imbalance > 5 else '허용 범위'}\n")

# ── 7.2 연속형 변수 단변량 분석 ──────────────────────────────────
valid_cont = [c for c in CONT_COLS if c in df_model.columns]
print("=" * 60)
print("  7.2 연속형 변수 기술 통계")
print("=" * 60)
desc = df_model[valid_cont].describe().T
desc["skewness"] = df_model[valid_cont].skew()
desc["kurtosis"] = df_model[valid_cont].kurt()
print(desc[["count", "mean", "std", "min", "50%", "max", "skewness", "kurtosis"]].round(2).to_string(), "\n")

fig, axes = plt.subplots(2, len(valid_cont), figsize=(14, 8))
for i, col in enumerate(valid_cont):
    data = df_model[col].dropna()
    axes[0][i].hist(data, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0][i].axvline(data.mean(),   color="red",    linestyle="--", label=f"평균 {data.mean():.1f}")
    axes[0][i].axvline(data.median(), color="orange", linestyle="--", label=f"중앙값 {data.median():.1f}")
    axes[0][i].set_title(col)
    axes[0][i].legend(fontsize=8)
    # palette를 dict 대신 리스트로 전달 (seaborn 최신 버전 호환)
    plot_df = df_model[[col, TARGET]].dropna().copy()
    plot_df[TARGET] = plot_df[TARGET].astype(str)
    sns.boxplot(data=plot_df, x=TARGET, y=col, ax=axes[1][i],
                order=["0", "1"], palette={"0": "steelblue", "1": "tomato"})
    axes[1][i].set_xticklabels(["비위험군", "위험군"])
    axes[1][i].set_title(f"{col} vs 위험군")
plt.suptitle("7.2 연속형 변수 분포 및 위험군 비교", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/02_continuous.png", dpi=150, bbox_inches="tight")
plt.show()

print("=== Mann-Whitney U 검정 (연속형 vs 위험군) ===")
for col in valid_cont:
    g0 = df_model[df_model[TARGET] == 0][col].dropna()
    g1 = df_model[df_model[TARGET] == 1][col].dropna()
    if len(g0) > 0 and len(g1) > 0:
        _, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
        sig = "★ 유의" if p < 0.05 else "비유의"
        print(f"  {col:<25}: 비위험군 {g0.mean():.2f} / 위험군 {g1.mean():.2f}  p={p:.4f}  {sig}")

In [ ]:
# ── 7.3 범주형 변수 분석 + 카이제곱 검정 ────────────────────────
valid_cat = [c for c in CAT_COLS if c in df_model.columns]
n_cols = 3
n_rows = -(-len(valid_cat) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(valid_cat):
    ct = pd.crosstab(df_model[col], df_model[TARGET], normalize="index") * 100
    ct.columns = ["비위험군", "위험군"]
    ct.plot(kind="bar", ax=axes[i], color=colors, rot=0, edgecolor="white")
    axes[i].set_title(col)
    axes[i].set_ylabel("비율(%)")
    axes[i].legend(fontsize=8)
for j in range(len(valid_cat), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("7.3 범주형 변수별 우울 위험군 비율(%)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/03_categorical.png", dpi=150, bbox_inches="tight")
plt.show()

print("=== 카이제곱 검정 (범주형 vs 위험군) ===")
for col in valid_cat:
    ct = pd.crosstab(df_model[col], df_model[TARGET])
    if ct.shape[1] < 2:
        continue
    chi2, p, _, _ = stats.chi2_contingency(ct)
    sig = "★ 유의" if p < 0.05 else "비유의"
    print(f"  {col:<25}: χ²={chi2:>7.2f}  p={p:.4f}  {sig}")

In [ ]:
# ── 7.4 상관관계 분석 ────────────────────────────────────────────
corr_cols = available_features + [TARGET]
corr = df_model[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=axes[0], linewidths=0.5, annot_kws={"size": 7})
axes[0].set_title("변수 간 상관관계 (하삼각)")

target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
bar_colors  = ["tomato" if v > 0 else "steelblue" for v in target_corr]
axes[1].barh(target_corr.index, target_corr.values, color=bar_colors, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("depression_risk와의 상관관계 (Pearson r)")
axes[1].set_xlabel("r")
plt.suptitle("7.4 상관관계 분석", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/04_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("=== 타깃과 상관관계 TOP 5 ===")
print(target_corr.head(5).round(3).to_string())

In [ ]:
# ── 7.5 이상치 탐지 (IQR) ────────────────────────────────────────
print("=" * 60)
print("  7.5 이상치 탐지 (IQR 1.5 기준, 연속형 변수)")
print("=" * 60)
outlier_info = {}
for col in valid_cont:
    q1, q3 = df_model[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df_model[col] < lo) | (df_model[col] > hi)).sum()
    outlier_info[col] = {"lower": lo, "upper": hi, "n": n_out, "rate": n_out / len(df_model)}
    print(f"  {col:<25}: 정상 범위 [{lo:.1f}, {hi:.1f}]  →  이상치 {n_out}개 ({n_out/len(df_model):.1%})")

# ── 7.6 결측치 분석 ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("  7.6 결측치 분석")
print("=" * 60)
missing = df_model[available_features].isna().mean().sort_values(ascending=False)
miss_df = df_model[available_features].isna().sum().sort_values(ascending=False).to_frame("결측 수")
miss_df["결측률"] = missing
print(miss_df.to_string(), "\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
missing_nz = missing[missing > 0]
if len(missing_nz) > 0:
    axes[0].barh(missing_nz.index, missing_nz.values * 100, color="tomato", alpha=0.8)
    axes[0].set_xlabel("결측률(%)")
    axes[0].set_title("변수별 결측률")
    for i, v in enumerate(missing_nz.values):
        axes[0].text(v * 100 + 0.3, i, f"{v:.1%}", va="center", fontsize=9)
else:
    axes[0].text(0.5, 0.5, "결측치 없음", ha="center", transform=axes[0].transAxes)
sns.heatmap(df_model[available_features].isna().T, cbar=False,
            cmap=["#f0f0f0", "tomato"], ax=axes[1],
            yticklabels=True, xticklabels=False)
axes[1].set_title("결측치 패턴 (빨간색=결측)")
plt.suptitle("7.6 결측치 분석", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/05_missing.png", dpi=150, bbox_inches="tight")
plt.show()

# ── EDA 요약 ────────────────────────────────────────────────────
top_out = max(outlier_info, key=lambda x: outlier_info[x]["rate"]) if outlier_info else "N/A"
print("=" * 60)
print("  EDA 요약")
print("=" * 60)
print(f"1. 클래스 불균형  : {imbalance:.1f}:1  → 모델링 시 stratify / class_weight 적용 권장")
print(f"2. 수면시간       : 위험군이 더 짧은 경향 (Mann-Whitney 검정 확인)")
print(f"3. 결측치 최다    : {missing.index[0]} ({missing.iloc[0]:.1%}) → 파이프라인 내 imputer로 처리")
print(f"4. 타깃 상관 1위  : {target_corr.index[0]} (r={target_corr.iloc[0]:.3f})")
if top_out != "N/A":
    print(f"5. 이상치 최다    : {top_out} ({outlier_info[top_out]['rate']:.1%}) → 파이프라인 내 처리 또는 클리핑 검토")

---

# 8. Train/Test Split 및 전처리 파이프라인

## 클래스 불균형 처리 전략

EDA 결과 비위험군 : 위험군 = **13.7 : 1**의 심각한 클래스 불균형이 확인됐다.
다음 두 가지 방법을 적용한다.

| 방법 | 적용 위치 | 효과 |
|------|-----------|------|
| `stratify=y` | `train_test_split` | 분할 후에도 클래스 비율 유지 |
| `class_weight='balanced'` | 분류 모델 초기화 | 소수 클래스(위험군)에 역비율 가중치 부여 |

결측치 대체, 인코딩, 스케일링은 train/test split 이후 파이프라인 내부에서 수행한다 (데이터 누수 방지).

In [ ]:
# hear_device_freq 원본 제거 (missing indicator로 대체), 엔지니어링 피처 포함
drop_from_X = [c for c in ["id", "depression_risk", "hear_device_freq"] if c in df_model.columns]

X = df_model.drop(columns=drop_from_X)
y = df_model["depression_risk"]

if y.nunique() < 2:
    raise ValueError("타깃 클래스가 하나뿐입니다. depression_risk 생성 기준을 확인하세요.")

# stratify=y: 클래스 불균형(13.7:1) 하에서 분할 후에도 비율 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("\nTrain target ratio")
print(y_train.value_counts(normalize=True).sort_index())
print("\nTest target ratio")
print(y_test.value_counts(normalize=True).sort_index())

In [ ]:
# 수치형: 연속형 + 이진 엔지니어링 피처 (스케일링 적용)
numeric_features = [c for c in [
    "wkdy_sleep_hours", "wknd_sleep_hours", "ear_noise_min", "tinnitus_dist",
    "sleep_diff", "sleep_short", "hear_device_freq_missing",
] if c in X.columns]

# 범주형: 순서형/명목형 설문 변수 (OneHotEncoding)
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

print("수치형 변수:", numeric_features)
print("범주형 변수:", categorical_features)